In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

FOLDERNAME = 'training_zen/phase2/assignment2_4'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content/drive/My\ Drive/$FOLDERNAME

In [ ]:
print(FOLDERNAME)

In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt
!pip install -q IPython==8.18.1
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import torch
import yaml
from PIL import Image

sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("dreamboth-lora-trainer"))

from models.stable_diffusion import StableDiffusion
from src.model_utils import inject_lora

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

sd = StableDiffusion()
sd.load_weight(
    model_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
    device=device,
    quantization="fp16" if device == "cuda" else "fp32"
)

with open("dreamboth-lora-trainer/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)
sd.unet, sd.text_encoder = inject_lora(sd.unet, sd.text_encoder, config)

lora_weights_path = "./checkpoints/final_lora_weights.safetensors"
if not os.path.exists(lora_weights_path):
    lora_weights_path = "./checkpoints/final_lora_weights.bin"

if os.path.exists(lora_weights_path):
    if lora_weights_path.endswith(".safetensors"):
        from safetensors.torch import load_file
        state_dict = load_file(lora_weights_path)
    else:
        state_dict = torch.load(lora_weights_path, map_location="cpu")
        
    unet_state_dict = {k.replace("unet.", ""): v for k, v in state_dict.items() if k.startswith("unet.")}
    text_state_dict = {k.replace("text_encoder.", ""): v for k, v in state_dict.items() if k.startswith("text_encoder.")}
    
    sd.unet.load_state_dict(unet_state_dict, strict=False)
    if len(text_state_dict) > 0 and sd.text_encoder is not None:
        sd.text_encoder.load_state_dict(text_state_dict, strict=False)


In [ ]:
def display_image(output_tensors,num):
    image_tensor = (output_tensors[0] / 2.0 + 0.5).clamp(0, 1)
    image_numpy = image_tensor.cpu().permute(1, 2, 0).numpy()
    image_numpy = (image_numpy * 255.0).astype("uint8")
    image = Image.fromarray(image_numpy)

    output_name = f"pic_{num}.png"
    image.save(output_name)


In [ ]:
noise = torch.randn(1, 4, 64, 64, device=device, dtype=torch_dtype)
prompt = "a photo of sks person"
negative_prompt = "blurry, low quality, distorted, bad anatomy"

with torch.no_grad():
    output_tensors = sd(
        prompts=[prompt],
        noise=noise,
        num_steps=50,
        device=device,
        guidance_scale=7.5,
        negative_prompts=[negative_prompt]
    )

# image_tensor = (output_tensors[0] / 2.0 + 0.5).clamp(0, 1)
# image_numpy = image_tensor.cpu().permute(1, 2, 0).numpy()
# image_numpy = (image_numpy * 255.0).astype("uint8")
# image = Image.fromarray(image_numpy)

# output_name = "sks_person_custom_generation.png"
# image.save(output_name)
image = display_image(output_tensors,1)

In [ ]:
image